# Physical real-slice retry v2 — repro first
CPU/high RAM, no GPU. Exactly one execution; no exploratory CI.
Repro → physical prerequisites → four draft audits. Not a cold seal;20/41 unchanged.


In [ ]:
import hashlib, pathlib, subprocess, sys, urllib.request, datetime, psutil
SOURCE_SHA = "59f9f522f3f731ac8a6270ac5c3ae719b1b201f6"
RUNNER_REV = "cmp99-physical-real-slice-retry-v2"
LAUNCHER_URL = "https://raw.githubusercontent.com/lluiseriksson/THE-ERIKSSON-PROGRAMME/7f3cc35e329b9f94ea432d861d78e54a56f8a005/scripts/launch_cmp99_physical_real_slice_retry.py"
LAUNCHER_SHA256 = "c9bb2647c4206b4ea263b0d94936dde9495a7ab2c53ff472d06e7b1f1497a0b9"
assert psutil.virtual_memory().total / 2**30 >= 40, "HIGH_RAM_REQUIRED"
assert not pathlib.Path("/dev/nvidia0").exists(), "GPU_NOT_AUTHORIZED"
launcher = pathlib.Path("/content/launch_physical_real_slice_retry_v2.py")
assert not launcher.exists(), "ALREADY_STARTED_NO_REEXECUTION"
with urllib.request.urlopen(LAUNCHER_URL, timeout=60) as response:
    payload = response.read()
assert hashlib.sha256(payload).hexdigest() == LAUNCHER_SHA256, "LAUNCHER_HASH_MISMATCH"
launcher.write_bytes(payload)
print("SOURCE_SHA=" + SOURCE_SHA + " RUNNER_REV=" + RUNNER_REV, flush=True)
print("HASH_GATE=PASS START_UTC=" + datetime.datetime.now(datetime.timezone.utc).isoformat(), flush=True)
with open("/content/physical-real-slice-retry-v2-console.log", "xb") as log:
    child = subprocess.Popen([sys.executable, "-u", str(launcher)], stdout=log, stderr=subprocess.STDOUT)
    print("LAUNCH_PID=" + str(child.pid), flush=True)
    code = child.wait()
pathlib.Path("/content/physical-real-slice-retry-v2-exit.txt").write_text(str(code) + "\n")
print(pathlib.Path("/content/physical-real-slice-retry-v2-console.log").read_text(errors="replace")[-12000:], flush=True)
print("LAUNCHER_EXIT=" + str(code), flush=True)
if code:
    raise RuntimeError("Diagnostic stopped: preserve first error; do not reexecute")
